# Phase 4 — Model training and comparison

Train **Naive Bayes** (baseline), **Logistic Regression**, and **linear SVM** (`LinearSVC`) on TF-IDF features. Record **training time** and **accuracy on the held-out test set** (same split and vectorizer as `feature_extraction.ipynb`). The **best** model by test accuracy is saved as **`models/best_model.pkl`** (use with `models/tfidf_vectorizer.pkl` at inference).

In [1]:
%pip install -q pandas scikit-learn scipy joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import time
from pathlib import Path

import joblib
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models"

PROCESSED_CSV = DATA_DIR / "processed_dataset.csv"
VECTORIZER_PATH = MODELS_DIR / "tfidf_vectorizer.pkl"
BEST_MODEL_PATH = MODELS_DIR / "best_model.pkl"

RANDOM_STATE = 42
TEST_SIZE = 0.20

In [3]:
if not VECTORIZER_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {VECTORIZER_PATH}. Run notebooks/feature_extraction.ipynb first."
    )

df = pd.read_csv(PROCESSED_CSV)
df = df.dropna(subset=["text_clean", "target"])
df["text_clean"] = df["text_clean"].astype(str).str.strip()
df = df[df["text_clean"].str.len() > 0]

X_raw = df["text_clean"]
y = df["target"].astype(int)
if not set(y.unique()).issubset({0, 1, 2}):
    raise ValueError("Expected target in {0,1,2}.")

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

vectorizer = joblib.load(VECTORIZER_PATH)
X_train = vectorizer.transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)

print("Train:", X_train.shape[0], " Test:", X_test.shape[0])
print("Feature matrix:", X_train.shape)

Train: 3600  Test: 900
Feature matrix: (3600, 69)


In [4]:
models_cfg: list[tuple[str, object]] = [
    ("Naive Bayes", MultinomialNB()),
    (
        "Logistic Regression",
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    ),
    (
        "SVM (LinearSVC)",
        LinearSVC(random_state=RANDOM_STATE, max_iter=8000),
    ),
]

results: list[dict] = []
fitted: dict[str, object] = {}

for name, clf in models_cfg:
    t0 = time.perf_counter()
    clf.fit(X_train, y_train)
    train_time_s = time.perf_counter() - t0
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results.append(
        {"model": name, "train_time_s": train_time_s, "validation_accuracy": acc}
    )
    fitted[name] = clf
    print(f"{name}: train_time={train_time_s:.4f}s  validation_acc={acc:.4f}")

summary = pd.DataFrame(results).sort_values(
    "validation_accuracy", ascending=False
)
display(summary)

Naive Bayes: train_time=0.0040s  validation_acc=1.0000
Logistic Regression: train_time=0.0177s  validation_acc=1.0000
SVM (LinearSVC): train_time=0.0262s  validation_acc=1.0000


,model,train_time_s,validation_accuracy
0,Naive Bayes,0.003999,1.0
1,Logistic Regression,0.017694,1.0
2,SVM (LinearSVC),0.026202,1.0


In [5]:
best_name = summary.iloc[0]["model"]
best_clf = fitted[best_name]
best_acc = float(summary.iloc[0]["validation_accuracy"])

joblib.dump(best_clf, BEST_MODEL_PATH)
print(f"Best model: {best_name}  (validation accuracy={best_acc:.4f})")
print("Saved:", BEST_MODEL_PATH)

Best model: Naive Bayes  (validation accuracy=1.0000)
Saved: D:\ml project\Social media and sentiment analysis\models\best_model.pkl
